In [1]:
import torch
print("PyTorch CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("CUDA device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device")

PyTorch CUDA available: True
CUDA device count: 1
CUDA device name: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [2]:
from transformers import AutoModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Example: Load a small model to test GPU
model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
print("Model loaded on device:", next(model.parameters()).device)

C:\Users\bchal\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Model loaded on device: cuda:0


In [3]:
# Import necessary libraries
import numpy as np
import requests
import sys
import random
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
assessment_questions = [
    {
        "question": "How would you rate your mood today?",
        "options": ["1 - Very bad", "2 - Bad", "3 - Neutral", "4 - Good", "5 - Very good"]
    },
    {
        "question": "Have you been enjoying activities you usually find pleasurable?",
        "options": ["Yes", "No"]
    },
    {
        "question": "How has your sleep been recently?",
        "options": ["Very poor", "Poor", "Average", "Good", "Very good"]
    },
    {
        "question": "How would you describe your energy levels?",
        "options": ["Very low", "Low", "Moderate", "High", "Very high"]
    },
    {
        "question": "How is your appetite lately?",
        "options": ["Very poor", "Poor", "Average", "Good", "Very good"]
    },
    {
        "question": "Have you been able to concentrate on tasks?",
        "options": ["Not at all", "Rarely", "Sometimes", "Often", "Always"]
    },
    {
        "question": "Do you often feel overwhelmed?",
        "options": ["Never", "Rarely", "Sometimes", "Often", "Always"]
    },
    {
        "question": "How would you describe your outlook on the future?",
        "options": ["Very negative", "Negative", "Neutral", "Positive", "Very positive"]
    },
    {
        "question": "Do you feel supported by friends and family?",
        "options": ["Not at all", "A little", "Somewhat", "Mostly", "Completely"]
    },
    {
        "question": "Have you had thoughts that life isn't worth living?",
        "options": ["Never", "Rarely", "Sometimes", "Often", "Always"]
    }
]


In [5]:
def ask_questions():
    responses = []
    print("Hello! I'm your MindCare-AI assistant.")
    print("Please answer the following questions by selecting the number of your choice.\n")
    for idx, q in enumerate(assessment_questions):
        print(f"Q{idx+1}: {q['question']}")
        for i, opt in enumerate(q['options']):
            print(f"  {i+1}. {opt}")
        while True:
            choice = input("Your choice (number): ").strip()
            if choice.isdigit() and 1 <= int(choice) <= len(q['options']):
                responses.append(q['options'][int(choice)-1])
                break
            else:
                print("Please enter a valid option number.")
        print()
    return responses

In [6]:
def assess_stress_level_manually(responses):
    """Fallback function to estimate stress level based on rule-based approach"""
    # Count negative indicators
    negative_count = 0
    
    # Check mood (first question)
    if "1 -" in responses[0] or "2 -" in responses[0]:
        negative_count += 2
    elif "3 -" in responses[0]:
        negative_count += 1
        
    # Check enjoyment (second question)
    if responses[1] == "No":
        negative_count += 2
        
    # Check sleep, energy, appetite (questions 3-5)
    for i in range(2, 5):
        if responses[i] in ["Very poor", "Poor"]:
            negative_count += 1
            
    # Check concentration (question 6)
    if responses[5] in ["Not at all", "Rarely"]:
        negative_count += 1
            
    # Check overwhelmed (question 7)
    if responses[6] in ["Often", "Always"]:
        negative_count += 1
            
    # Check outlook (question 8)
    if responses[7] in ["Very negative", "Negative"]:
        negative_count += 1
            
    # Check support (question 9)
    if responses[8] in ["Not at all", "A little"]:
        negative_count += 1
            
    # Check thoughts (question 10)
    if responses[9] in ["Sometimes", "Often", "Always"]:
        negative_count += 2
            
    # Convert to stress level
    if negative_count >= 9:
        return 5
    elif negative_count >= 7:
        return 4
    elif negative_count >= 5:
        return 3
    elif negative_count >= 3:
        return 2
    else:
        return 1

In [7]:
def get_stress_level_with_ollama_api(responses):
    """Uses the Ollama API directly instead of subprocess"""
    api_url = "http://127.0.0.1:11434/api/generate"
    
    # Prepare the prompt
    prompt = (
        "Given the following answers to a mental health assessment, "
        "estimate the user's overall stress level on a scale from 1 (very low) to 5 (very high). "
        "Only return the number (1-5) as your answer.\n\n"
        "Answers:\n"
    )
    for i, (q, a) in enumerate(zip(assessment_questions, responses)):
        prompt += f"{i+1}. {q['question']} Answer: {a}\n"
    prompt += "\nStress level (1-5):"

    # Try different models in order of preference
    for model_name in ["gemma3:4b", "qwen3:1.7b", "mistral:7b", "llama2:7b"]:
        try:
            print(f"Trying with model: {model_name}...")
            
            # Prepare the API request
            payload = {
                "model": model_name,
                "prompt": prompt,
                "stream": False,
                "temperature": 0.1  # Low temperature for more deterministic output
            }
            
            # Make the API request
            response = requests.post(api_url, json=payload, timeout=30)
            
            # Check if request was successful
            if response.status_code == 200:
                result = response.json()
                output = result.get("response", "").strip()
                print(f"Model response: {output}")
                
                # Extract the first digit 1-5 from the output
                for c in output:
                    if c in "12345":
                        return int(c)
                
                # If we got output but no valid number, try next model
                print("No valid stress level found in response.")
            else:
                print(f"API request failed with status code: {response.status_code}")
                
        except requests.exceptions.Timeout:
            print(f"Request to model {model_name} timed out.")
        except requests.exceptions.ConnectionError:
            print(f"Connection error. Is Ollama running at {api_url}?")
            break  # Break the loop if Ollama is not running
        except Exception as e:
            print(f"Error with {model_name}: {e}")
    
    # If all models failed, use rule-based approach
    print("Using rule-based assessment as fallback.")
    return assess_stress_level_manually(responses)


In [8]:
def get_personalized_advice_with_ollama(responses, stress_level):
    """Get personalized advice from Ollama based on user's specific responses"""
    api_url = "http://127.0.0.1:11434/api/generate"
    
    # Create a summary of concerning responses
    concerning_responses = []
    
    # Check for concerning responses
    if "1 -" in responses[0] or "2 -" in responses[0]:
        concerning_responses.append(f"You rated your mood as {responses[0]}")
    
    if responses[1] == "No":
        concerning_responses.append("You haven't been enjoying activities you usually find pleasurable")
    
    for i in range(2, 5):
        if responses[i] in ["Very poor", "Poor"]:
            concerning_responses.append(f"{assessment_questions[i]['question']}: {responses[i]}")
    
    if responses[5] in ["Not at all", "Rarely"]:
        concerning_responses.append(f"You've been having trouble concentrating")
    
    if responses[6] in ["Often", "Always"]:
        concerning_responses.append(f"You feel overwhelmed {responses[6].lower()}")
    
    if responses[7] in ["Very negative", "Negative"]:
        concerning_responses.append(f"Your outlook on the future is {responses[7].lower()}")
    
    if responses[8] in ["Not at all", "A little"]:
        concerning_responses.append(f"You don't feel very supported by friends and family")
    
    if responses[9] in ["Sometimes", "Often", "Always"]:
        concerning_responses.append(f"You've had thoughts that life isn't worth living {responses[9].lower()}")
    
    # Prepare the prompt
    prompt = f"""You are a compassionate mental health assistant. The user has completed a mental health assessment 
and their stress level is {stress_level}/5 (where 5 is highest stress).

Their concerning responses include:
- {'\n- '.join(concerning_responses if concerning_responses else ['None specifically'])}

Please provide personalized, actionable advice to help them manage their stress and improve their mental wellbeing.
Give 3-5 specific suggestions that address their particular concerns.
For very high stress levels (4-5), recommend professional help but also provide immediate coping strategies.
Keep your response caring, positive and supportive. Use about 150-200 words."""

    # Try different models in order of preference
    for model_name in ["gemma3:4b", "qwen3:1.7b", "mistral:7b", "llama2:7b"]:
        try:
            print(f"Getting advice using model: {model_name}...")
            
            # Prepare the API request
            payload = {
                "model": model_name,
                "prompt": prompt,
                "stream": False,
                "temperature": 0.7  # Slightly higher temperature for more creative advice
            }
            
            # Make the API request
            response = requests.post(api_url, json=payload, timeout=45)
            
            # Check if request was successful
            if response.status_code == 200:
                result = response.json()
                advice = result.get("response", "").strip()
                
                if len(advice) > 30:  # Make sure we got a meaningful response
                    return advice
                
            else:
                print(f"API request failed with status code: {response.status_code}")
                
        except Exception as e:
            print(f"Error getting advice with {model_name}: {e}")
    
    # Fallback advice if all models fail
    return get_fallback_advice(stress_level)

def get_fallback_advice(stress_level):
    """Return fallback advice based on stress level if API calls fail"""
    if stress_level <= 2:
        return "Your stress level appears relatively low. Continue your healthy habits and self-care routines. Regular exercise, good sleep, and social connections all contribute to maintaining positive mental health. Take a moment each day to appreciate what's going well in your life."
    elif stress_level == 3:
        return "You're experiencing moderate stress. Try incorporating regular breaks into your day, practice deep breathing when feeling overwhelmed, and ensure you're making time for activities you enjoy. Limiting screen time before bed and maintaining a consistent sleep schedule can also help manage stress levels."
    else:
        return "Your stress level is high. Consider talking to a trusted friend or mental health professional about how you're feeling. In the meantime, try stress-reduction techniques like mindfulness meditation, physical exercise, or journaling. Remember to be kind to yourself and recognize when you need to set boundaries. Your mental health is important, and seeking support is a sign of strength."

In [9]:
def run_chatbot():
    responses = ask_questions()
    print("Calculating your stress level using AI...\n")
    
    # Use API instead of subprocess
    stress_level = get_stress_level_with_ollama_api(responses)
    
    if stress_level:
        print(f"Your estimated stress level is: {stress_level}/5")
        
        # Get personalized advice from the AI
        print("\nGenerating personalized recommendations...\n")
        advice = get_personalized_advice_with_ollama(responses, stress_level)
        print(advice)
        
        # If stress level is very high (4-5) and there are concerning thoughts about life
        if stress_level >= 4 and responses[9] in ["Sometimes", "Often", "Always"]:
            print("\nIMPORTANT: If you're having thoughts that life isn't worth living, please reach out for help.")
            print("Crisis resources: National Suicide Prevention Lifeline: 1-800-273-8255")
            print("Or text HOME to 741741 to reach the Crisis Text Line.")
            print("Your life matters, and support is available.")
    else:
        print("Sorry, I couldn't determine your stress level. Please try again later.")

In [10]:
def evaluate_chatbot_accuracy(num_samples=50):
    """
    Evaluate the accuracy and precision of the mental health assessment.
    
    Uses a comparison between AI predictions and expert ratings (simulated for demo).
    """
    
    
    # Function to generate simulated responses (for testing only)
    def generate_test_response():
        simulated_responses = []
        # Generate random responses, biased toward consistency
        mood_level = random.randint(1, 5)
        # Mood follows the 1-5 distribution
        simulated_responses.append(f"{mood_level} - {'Very bad' if mood_level == 1 else 'Bad' if mood_level == 2 else 'Neutral' if mood_level == 3 else 'Good' if mood_level == 4 else 'Very good'}")
        
        # Low mood tends to correlate with other negative responses
        if mood_level <= 2:
            simulated_responses.append(random.choices(["Yes", "No"], weights=[20, 80])[0])
        else:
            simulated_responses.append(random.choices(["Yes", "No"], weights=[80, 20])[0])
        
        # Sleep, energy, appetite
        for _ in range(3):
            if mood_level <= 2:
                simulated_responses.append(random.choices(["Very poor", "Poor", "Average", "Good", "Very good"], 
                                                       weights=[30, 40, 20, 7, 3])[0])
            else:
                simulated_responses.append(random.choices(["Very poor", "Poor", "Average", "Good", "Very good"], 
                                                       weights=[5, 15, 30, 30, 20])[0])
        
        # Concentration
        if mood_level <= 3:
            simulated_responses.append(random.choices(["Not at all", "Rarely", "Sometimes", "Often", "Always"], 
                                                   weights=[20, 30, 30, 15, 5])[0])
        else:
            simulated_responses.append(random.choices(["Not at all", "Rarely", "Sometimes", "Often", "Always"], 
                                                   weights=[5, 15, 30, 30, 20])[0])
        
        # Overwhelmed
        if mood_level <= 3:
            simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                  weights=[5, 10, 25, 35, 25])[0])
        else:
            simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                  weights=[25, 35, 25, 10, 5])[0])
        
        # Future outlook
        if mood_level <= 2:
            simulated_responses.append(random.choices(["Very negative", "Negative", "Neutral", "Positive", "Very positive"], 
                                                  weights=[25, 45, 20, 7, 3])[0])
        else:
            simulated_responses.append(random.choices(["Very negative", "Negative", "Neutral", "Positive", "Very positive"], 
                                                  weights=[3, 7, 20, 45, 25])[0])
        
        # Support
        simulated_responses.append(random.choices(["Not at all", "A little", "Somewhat", "Mostly", "Completely"])[0])
        
        # Suicidal thoughts - treat with appropriate care in simulation
        if mood_level == 1:
            simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                  weights=[40, 25, 20, 10, 5])[0])
        else:
            simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                  weights=[70, 20, 8, 1, 1])[0])
            
        return simulated_responses
    
    # For real evaluation, you would have expert-rated data
    # Here we'll simulate that by generating random test cases
    
    # Lists to store results
    ai_predictions = []
    rule_based_predictions = []
    expert_ratings = []  # This would come from human experts in real evaluation
    
    print(f"Evaluating chatbot on {num_samples} simulated cases...")
    
    for i in range(num_samples):
        # Generate a simulated response set
        test_responses = generate_test_response()
        
        # Get AI prediction
        ai_stress_level = get_stress_level_with_ollama_api(test_responses)
        
        # Get rule-based prediction
        rule_stress_level = assess_stress_level_manually(test_responses)
        
        # Simulate expert rating (in reality, this would be done by professionals)
        # For simulation, we'll make the expert rating somewhat correlated with the rule-based approach
        # but with some variation to simulate differences in clinical judgment
        variation = random.choices([-1, 0, 1], weights=[20, 60, 20])[0]
        expert_level = max(1, min(5, rule_stress_level + variation))
        
        # Store results
        ai_predictions.append(ai_stress_level)
        rule_based_predictions.append(rule_stress_level)
        expert_ratings.append(expert_level)
        
        print(f"Case {i+1}: AI={ai_stress_level}, Rule={rule_stress_level}, Expert={expert_level}")
    
    # Calculate metrics
    ai_accuracy = accuracy_score(expert_ratings, ai_predictions)
    rule_accuracy = accuracy_score(expert_ratings, rule_based_predictions)
    
    # Calculate precision for each stress level
    ai_precision_macro = precision_score(expert_ratings, ai_predictions, average='macro', zero_division=0)
    rule_precision_macro = precision_score(expert_ratings, rule_based_predictions, average='macro', zero_division=0)
    
    # Calculate recall
    ai_recall = recall_score(expert_ratings, ai_predictions, average='macro', zero_division=0)
    rule_recall = recall_score(expert_ratings, rule_based_predictions, average='macro', zero_division=0)
    
    # Generate confusion matrices
    ai_cm = confusion_matrix(expert_ratings, ai_predictions, labels=[1, 2, 3, 4, 5])
    rule_cm = confusion_matrix(expert_ratings, rule_based_predictions, labels=[1, 2, 3, 4, 5])
    
    # Print results
    print("\n===== CHATBOT EVALUATION RESULTS =====")
    print(f"Number of test cases: {num_samples}")
    print("\nAI Model Performance:")
    print(f"Accuracy: {ai_accuracy:.2f}")
    print(f"Precision (macro): {ai_precision_macro:.2f}")
    print(f"Recall (macro): {ai_recall:.2f}")
    print(f"F1 Score: {2 * (ai_precision_macro * ai_recall) / (ai_precision_macro + ai_recall) if (ai_precision_macro + ai_recall) > 0 else 0:.2f}")
    
    print("\nRule-based Model Performance:")
    print(f"Accuracy: {rule_accuracy:.2f}")
    print(f"Precision (macro): {rule_precision_macro:.2f}")
    print(f"Recall (macro): {rule_recall:.2f}")
    print(f"F1 Score: {2 * (rule_precision_macro * rule_recall) / (rule_precision_macro + rule_recall) if (rule_precision_macro + rule_recall) > 0 else 0:.2f}")
    
    # Plot confusion matrices
    plt.figure(figsize=(15, 7))
    
    plt.subplot(1, 2, 1)
    sns.heatmap(ai_cm, annot=True, fmt='d', cmap='Blues', xticklabels=[1, 2, 3, 4, 5], yticklabels=[1, 2, 3, 4, 5])
    plt.title('AI Model Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    
    plt.subplot(1, 2, 2)
    sns.heatmap(rule_cm, annot=True, fmt='d', cmap='Blues', xticklabels=[1, 2, 3, 4, 5], yticklabels=[1, 2, 3, 4, 5])
    plt.title('Rule-based Model Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    
    plt.tight_layout()
    plt.savefig('chatbot_evaluation.png')
    plt.show()
    
    return {
        'ai_accuracy': ai_accuracy,
        'rule_accuracy': rule_accuracy,
        'ai_precision': ai_precision_macro,
        'rule_precision': rule_precision_macro,
        'ai_recall': ai_recall,
        'rule_recall': rule_recall
    }


In [11]:
if __name__ == "__main__":
    
    if len(sys.argv) > 1 and sys.argv[1] == "--evaluate":
        # Run evaluation mode
        evaluate_chatbot_accuracy(num_samples=20)  # Smaller sample size for demonstration
    else:
        # Run normal chatbot mode
        run_chatbot()

Hello! I'm your MindCare-AI assistant.
Please answer the following questions by selecting the number of your choice.

Q1: How would you rate your mood today?
  1. 1 - Very bad
  2. 2 - Bad
  3. 3 - Neutral
  4. 4 - Good
  5. 5 - Very good
Please enter a valid option number.

Q2: Have you been enjoying activities you usually find pleasurable?
  1. Yes
  2. No

Q3: How has your sleep been recently?
  1. Very poor
  2. Poor
  3. Average
  4. Good
  5. Very good

Q4: How would you describe your energy levels?
  1. Very low
  2. Low
  3. Moderate
  4. High
  5. Very high

Q5: How is your appetite lately?
  1. Very poor
  2. Poor
  3. Average
  4. Good
  5. Very good

Q6: Have you been able to concentrate on tasks?
  1. Not at all
  2. Rarely
  3. Sometimes
  4. Often
  5. Always

Q7: Do you often feel overwhelmed?
  1. Never
  2. Rarely
  3. Sometimes
  4. Often
  5. Always

Q8: How would you describe your outlook on the future?
  1. Very negative
  2. Negative
  3. Neutral
  4. Positive
  